# localml quickstart

A guided tour of the `localml` SDK against a running control plane. Bring the stack up first (`docker compose up -d`, or run the API standalone against SQLite), then work top to bottom.

You'll configure the client, track a run, register a model + dataset + prompt, run the predict → evaluate loop, and deploy the model behind the OpenAI-compatible serving proxy.

In [ ]:
import localml as ml

ml.configure(api_url="http://localhost:8000", token="local-dev-token")
ml.configure  # env vars / ~/.localml/config.toml also work

## 1. Track a run

A run groups params, metrics, and artifacts.

In [ ]:
with ml.start_run(project="local", config={"model": "tiny-llm"}) as run:
    ml.log_params({"batch_size": 4, "quantization": "4bit"})
    ml.log_metrics({"baseline_accuracy": 0.82})

run.id, run.status

## 2. Register a model version

Framework adapters (`ml.torch` / `ml.mlx` / `ml.huggingface` / `ml.jax`) package a model directory into a checksummed bundle and register it. Here we register a bare artifact URI directly.

In [ ]:
version = ml.register_model(
    name="tiny-assistant",
    artifact_uri="models/tiny-assistant",
    framework="mlx",
    metadata={"task": "chat"},
)
version.name, version.version, version.status

## 3. Register a dataset and a prompt

Datasets get stable per-row `example_id`s (needed later for comparison). Prompts are versioned `str.format` templates; the declared `{variables}` are pre-flighted against the dataset's columns before any inference runs.

In [ ]:
rows = [
    {"question": "What is a model registry?", "answer": "A versioned model store."},
    {"question": "What is a run?", "answer": "A tracked experiment execution."},
]
dataset = ml.datasets.register(
    project="local", name="evalset", artifact_uri="datasets/eval.jsonl", rows=rows
)
prompt = ml.prompts.register(name="qa", template="Q: {question}\nA:")
dataset.version, prompt.version, prompt.variables

## 4. Predict, then evaluate

`ml.evaluate(..., prompt=...)` is predict-then-eval sugar: it runs a prediction job, waits for it, then scores the stored results. Use the deterministic `echo` provider for a smoke test, or point `provider="openai"` at a local backend (`config={"base_url": ...}`).

In [ ]:
eval_job = ml.evaluate(
    version, dataset, ["exact_match", "latency_p95"], prompt=prompt, provider="echo"
)
eval_job.wait()
eval_job.metrics

## 5. Deploy and infer

A deployment resolves to an OpenAI-compatible backend and forwards requests through the proxy. Register a named provider, deploy, and round-trip a prompt. (Requires a local backend such as Ollama listening on `base_url`.)

In [ ]:
ml.providers.register("my-ollama", base_url="http://localhost:11434", model="llama3")
deployment = ml.deploy(version, provider="my-ollama")
deployment.predict({"prompt": "Explain model registries simply."})

## Next

See **`predict_eval_loop.ipynb`** for the core iteration workflow: register two prompt versions, predict with both, and compare to pick the winner.